In [159]:
import pandas as pd

In [160]:
input_df = pd.read_csv('../data/challenge_425.csv').reset_index(names=['Person'])

input_df

,Person,Amount
0,0,720
1,1,680
2,2,510
3,3,550
4,4,1000


In [161]:
# create df of weeks

weeks = [1, 2, 3, 4]

weeks_df = pd.DataFrame([{'week': x} for x in weeks])

weeks_df

,week
0,1
1,2
2,3
3,4


In [162]:
# cross join weeks and amounts

df = pd.merge(input_df, weeks_df, how='cross')
df['weekly max'] = 150

df.head()

,Person,Amount,week,weekly max
0,0,720,1,150
1,0,720,2,150
2,0,720,3,150
3,0,720,4,150
4,1,680,1,150


In [163]:
# calculate cumulative spend

df['cumulative spend'] = df.groupby('Person')['weekly max'].cumsum()

for i in range(len(df)):
    if df.loc[i, 'cumulative spend'] > df.loc[i, 'Amount']:
        df.loc[i, 'cumulative spend'] = df.loc[i, 'Amount']

df.head()

,Person,Amount,week,weekly max,cumulative spend
0,0,720,1,150,150
1,0,720,2,150,300
2,0,720,3,150,450
3,0,720,4,150,600
4,1,680,1,150,150


In [164]:
# calculate total spend

df['spend'] = 0

for i in range(len(df)):
    if i == 0 or (df.loc[i, 'Person'] != df.loc[i-1, 'Person']):
        df.loc[i, 'spend'] = min(150, df.loc[i, 'Amount'])
    elif df.loc[i, 'Person'] == df.loc[i-1, 'Person']:
        df.loc[i, 'spend'] = df.loc[i, 'cumulative spend'] - df.loc[i-1, 'cumulative spend']

df.head()

,Person,Amount,week,weekly max,cumulative spend,spend
0,0,720,1,150,150,150
1,0,720,2,150,300,150
2,0,720,3,150,450,150
3,0,720,4,150,600,150
4,1,680,1,150,150,150


In [165]:
# calculate the amount carried forward by each person and join it back to the original df

df['carry forward'] = df['Amount'] - df['cumulative spend']
idx = df.groupby('Person')['carry forward'].idxmin()

df = pd.merge(df.drop(['carry forward', 'weekly max'], axis=1), df.loc[idx][['Person', 'carry forward']], how='inner', on='Person')

df.head()

,Person,Amount,week,cumulative spend,spend,carry forward
0,0,720,1,150,150,120
1,0,720,2,300,150,120
2,0,720,3,450,150,120
3,0,720,4,600,150,120
4,1,680,1,150,150,80


In [166]:
# pivot so each week has its spend as an individual column

df.pivot(columns=['week'], values=['spend'], index=['Person', 'Amount', 'carry forward'])

spend               
week                            1    2    3    4
Person Amount carry forward                     
0      720    120             150  150  150  150
1      680    80              150  150  150  150
2      510    0               150  150  150   60
3      550    0               150  150  150  100
4      1000   400             150  150  150  150